TABLEAU A CONSTRUIRE

* lignes : INDEX 1 = dataset/model ; INDEX 2 = $\kappa_f$ ; INDEX 3 = 'hyperparameter value': n=... puis imbalance=... puis $\delta$=... 

* colonnes = normalized Areas Between Curves (ABC, between bounds curve and test metric curve, normalized by number of common $\theta$ points on x-axis): R0/1, R0/1, FPP, FPP, ..., moyennes calculées sur 10 splits de CV



In [15]:
import os
import sys
current_dir = os.getcwd()
root_path = os.path.abspath(os.path.join(current_dir, '..'))
if root_path not in sys.path:
    sys.path.append(root_path)
import torch
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import torch.nn as nn
import numpy as np
from IPython.display import clear_output
import pickle
import pandas as pd
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import math
import scipy.special
import random as rd
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import VGG16_Weights
from tqdm import tqdm
import torch.optim.lr_scheduler as lr_scheduler
from python_scripts.sgr_utils import *
from python_scripts.plotting import *
from python_scripts.preprocessing import *
from scipy.special import gammaln
import warnings
warnings.filterwarnings("ignore")
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 150

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: False


In [8]:
datasets_paths = [
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/BC/sgr_set_log_reg',
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/CIFAR2/sgr_set_cnn',
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/CIFAR2/sgr_set_resnet',
    #'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/CIFAR2/sgr_set_cnn_MCD',
    # put resnet_MCD path here
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/WSI/sgr_set_cnn'
]

In [ ]:
dicos = []
datasets = ['BC', 'CIFAR2', 'WSI']
models = ["log_reg", "cnn", "resnet"]

for path in datasets_paths:

    print('Considering ', path)
    mod = models[int(np.where([(m in path) for m in models])[0][0])]
    ds = datasets[int(np.where([(d in path) for d in datasets])[0][0])]
    if 'MCD' in path:
        kappa = 'MCD'
    else:
        kappa = 'SR'

    original_ds = pickle.load(open(path, 'rb'))

    print('Sample size proportions...')
    for prop in [1, 3/4, 1/2, 1/4]:
        print('prop = ', prop)
        for metric in ['standard', 'FP', 'FN', 'FPR', 'FNR']:
            print("metric = ", metric)
            values = []
            for seed in tqdm(range(10)):
                df = original_ds.sample(frac=prop, random_state=seed)
                values.append(ABC(df, metric=metric))
            dicos.append({'dataset': ds,
                            'model': mod,
                            'kappa': kappa,
                            'metric': metric,
                            'param.': 'N',
                            'value': prop,
                            'ABC': np.mean(values),
                            'CI': [np.mean(values)-1.96*np.std(values),
                                np.mean(values)+1.96*np.std(values)]})

    print('Imbalance rates...')
    for imbalance in [3/4, 1/2, 1/4]:
        print('imbalance = ', imbalance)
        for metric in ['standard', 'FP', 'FN', 'FPR', 'FNR']:
            print('metric = ', metric)
            values = []
            for seed in tqdm(range(10)):
                df = generate_imbalanced_datasets(original_ds, [imbalance], seed=seed)[0]
                values.append(ABC(df, metric=metric))
            dicos.append({'dataset': ds,
                        'model': mod,
                        'kappa': kappa,
                        'metric': metric,
                        'param.': 'imbalance',
                        'value': imbalance,
                        'ABC': np.mean(values),
                        'CI': [np.mean(values)-1.96*np.std(values),
                                np.mean(values)+1.96*np.std(values)]})
            
    print(r'$\delta$ values...')
    for delta in [0.001, 0.01, 0.1]:
        print('delta = ', delta)
        for metric in ['standard', 'FP', 'FN', 'FPR', 'FNR']:
            print('metric = ', metric)
            values = []
            for seed in tqdm(range(10)):
                df = original_ds.sample(frac=1, random_state=seed)
                values.append(ABC(df, metric=metric, delta=delta))
            dicos.append({'dataset': ds,
                            'model': mod,
                            'kappa': kappa,
                            'metric': metric,
                            'param.': 'delta',
                            'value': delta,
                            'ABC': np.mean(values),
                            'CI': [np.mean(values)-1.96*np.std(values),
                                np.mean(values)+1.96*np.std(values)]})
            
    clear_output(wait=True)

In [16]:
res = pd.DataFrame(dicos)
display(res)

,dataset,model,kappa,metric,param.,value,ABC,CI
0,BC,log_reg,SR,standard,N,1.0,0.059911,"[0.035529038838667676, 0.08429264414161042]"
1,BC,log_reg,SR,FP,N,1.0,0.050801,"[0.03554820939041309, 0.06605450016191074]"
2,BC,log_reg,SR,FN,N,1.0,0.056189,"[0.027689001840426046, 0.08468908986297369]"
3,BC,log_reg,SR,FPR,N,1.0,0.139155,"[0.12200717651879048, 0.15630275287109727]"
4,BC,log_reg,SR,FNR,N,1.0,0.376970,"[0.232008340256462, 0.5219313144169441]"
...,...,...,...,...,...,...,...,...
195,WSI,cnn,SR,standard,delta,0.1,0.005863,"[-0.0007718370847239015, 0.012497193246579934]"
196,WSI,cnn,SR,FP,delta,0.1,0.004533,"[-0.0004343107806395116, 0.009500572387194501]"
197,WSI,cnn,SR,FN,delta,0.1,0.003446,"[-0.00044373704777562247, 0.007335574323904152]"
198,WSI,cnn,SR,FPR,delta,0.1,0.008992,"[-0.0007171151384670425, 0.018702087580433006]"


In [ ]:
pickle.dump(res, open('hyperparams_impact_res', 'wb'))
res.to_csv('hyperparams_impact.csv')